In [1]:
import os 
import pandas as pd

In [74]:
def get_mean_interval(df, dire_var):
    df["service_date"] = pd.to_datetime(df["service_date"])
    df["year"] = df["service_date"].dt.year
    df["month"] = df["service_date"].dt.month_name()
    df['scheduled_datetime'] = pd.to_datetime(df['scheduled_datetime']) 
   
    df['actual_datetime'] = pd.to_datetime(df['actual_datetime'])
    df['scheduled_time_interval'] = df.groupby(['year', 'month', 'service_date', 'route_id', dire_var, 'stop_id'])['scheduled_datetime'].diff().dt.total_seconds()
    df['time_interval'] = df.groupby(['year', 'month', 'service_date', 'route_id', dire_var, 'stop_id'])['actual_datetime'].diff().dt.total_seconds()
    df['headway_latency'] = df['time_interval'] - df['scheduled_time_interval']
    ret = df.groupby(['year', 'month', 'stop_id'], as_index=False).agg(mean_headway_latency=('headway_latency', 'mean'), median_headway_latency=('headway_latency', 'median'))
    
    return ret.dropna()

In [37]:
files = os.listdir("../dataset-documentation/cleaned_data/arrival_depart_cleaned_data/")
files = sorted(files)
files

['cleaned_2022-10_data.csv',
 'cleaned_2022-11_data.csv',
 'cleaned_2019-04-06_data.csv',
 'cleaned_2022-07_data.csv',
 'cleaned_2022-06_data.csv',
 'cleaned_2019-10-12_data.csv',
 'cleaned_2022-01_data.csv',
 'cleaned_2022-04_data.csv',
 'cleaned_2022-05_data.csv',
 'cleaned_2022-12_data.csv',
 'cleaned_2019-07-09_data.csv',
 'cleaned_2018-1012_data.csv',
 'cleaned_2022-09_data.csv',
 'cleaned_2022-08_data.csv',
 'cleaned_2019-01-03_data.csv',
 'cleaned_2022-03_data.csv',
 'cleaned_2022-02_data.csv']

In [82]:
latency_df = pd.DataFrame()
for f in files:
    if "2018" in f: continue
    df = pd.read_csv(f'../dataset-documentation/cleaned_data/arrival_depart_cleaned_data/{f}')
    year = f[8:12]
    month = f[13:15]
    print(year, month)
    dire_var = "direction" if year == '2019' else "direction_id"
    t = get_mean_interval(df, dire_var)
    latency_df = pd.concat([latency_df, t], axis=0).reset_index(drop=True)
    

2022 10
2022 11
2019 04
2022 07
2022 06
2019 10
2022 01
2022 04
2022 05
2022 12
2019 07
2022 09
2022 08
2019 01
2022 03
2022 02


In [88]:
latency_df.to_csv("../dataset-documentation/cleaned_data/stop_headway_latency.csv", index=False)

In [95]:
latency_df["period"] = latency_df["year"].astype(str) + latency_df["month"]

In [109]:
count = latency_df.groupby("stop_id").nunique()["year"]
complete_stop_id = count.index[count == 2]

In [122]:
mean_latency_df = (
    latency_df.query("stop_id in @complete_stop_id")
    .groupby(["year", "stop_id"], as_index=False)
    .agg(mean_latency = ("mean_headway_latency", "mean"), count=("mean_headway_latency", "size"))

)

In [125]:
mean_latency_df.to_csv("../dataset-documentation/cleaned_data/mean_stop_latency.csv", index=False)